In [5]:
import pandas as pd

# Load all three raw files
train = pd.read_csv("/content/drive/MyDrive/DSPL_GCW/Hotel-A-train.csv")
val   = pd.read_csv("/content/drive/MyDrive/DSPL_GCW/Hotel-A-validation.csv")
test  = pd.read_csv("/content/drive/MyDrive/DSPL_GCW/Hotel-A-test.csv")

print("BEFORE CLEANING:")
print(f" Train      : {len(train):,} rows")
print(f" Validation : {len(val):,} rows")
print(f" Test       : {len(test):,} rows")


# Cleaning function
def clean_dataset(df, name):
    df = df.copy()

    # Parse dates
    for col in ["Booking_date", "Expected_checkin", "Expected_checkout"]:
        df[col] = pd.to_datetime(df[col], format="%m/%d/%Y", errors="coerce")

    # Calculate lead time
    df["lead_time"] = (df["Expected_checkin"] - df["Booking_date"]).dt.days

    # Remove rows where booking date is after check-in
    before = len(df)
    df = df[df["lead_time"] >= 0].copy()
    removed = before - len(df)

    # Standardise Reservation_Status
    # Unifies Check-In / Check-out to Check-Out
    if "Reservation_Status" in df.columns:
        df["Reservation_Status"] = df["Reservation_Status"].replace({
            "Check-In"  : "Check-Out",
            "Check-out" : "Check-Out",
        })

    print(f"\n{name}:")
    print(f"  Removed {removed:,} invalid rows (booking after check-in)")
    print(f"  Remaining : {len(df):,} rows")
    if "Reservation_Status" in df.columns:
        print(f"Status breakdown:")
        for status, count in df["Reservation_Status"].value_counts().items():
            print(f"    {status:<12} : {count:,}")

    return df

# Clean all three
train_clean = clean_dataset(train,"TRAIN")
val_clean   = clean_dataset(val,"VALIDATION")
test_clean  = clean_dataset(test,"TEST")

# Save cleaned files
train_clean.to_csv("train_cleaned.csv", index=False)
val_clean.to_csv("validation_cleaned.csv", index=False)
test_clean.to_csv("test_cleaned.csv", index=False)

print("\n")
print("AFTER CLEANING:")
print(f"Train      : {len(train_clean):,} rows - train_cleaned.csv")
print(f"Validation : {len(val_clean):,} rows - validation_cleaned.csv")
print(f"Test       : {len(test_clean):,} rows - test_cleaned.csv")
print("All three files cleaned and saved")

BEFORE CLEANING:
 Train      : 27,499 rows
 Validation : 2,749 rows
 Test       : 4,318 rows

TRAIN:
  Removed 506 invalid rows (booking after check-in)
  Remaining : 26,993 rows
Status breakdown:
    Check-Out    : 20,777
    Canceled     : 4,108
    No-Show      : 2,108

VALIDATION:
  Removed 16 invalid rows (booking after check-in)
  Remaining : 2,733 rows
Status breakdown:
    Check-Out    : 1,602
    Canceled     : 738
    No-Show      : 393

TEST:
  Removed 27 invalid rows (booking after check-in)
  Remaining : 4,291 rows


AFTER CLEANING:
Train      : 26,993 rows - train_cleaned.csv
Validation : 2,733 rows - validation_cleaned.csv
Test       : 4,291 rows - test_cleaned.csv
All three files cleaned and saved
